# 🚀 TASK 6.3-R: ANCHORED RECOVERY TRAINING TRÊN GOOGLE COLAB GPU
### CHIẾN DỊCH HUẤN LUYỆN NEO PHỤC HỒI HIỆU NĂNG ĐỈNH CAO — PHASE 6-R

> **Mã nhiệm vụ**: `Task 6.3-R` *(Milestone 6-R: Anchored Recovery)*  
> **Đơn vị chủ trì**: **Thành viên C (Training & Evaluation Lead)**  
> **Đơn vị phối hợp**: **Thành viên B (Architecture Lead)**, **Thành viên A (Data & Infra Lead)**  
> **Môi trường thực thi**: Google Colab GPU (Tesla T4 / L4 / A100 SXM4) + Google Drive 5TB  
> **Nguyên lý khôi phục đỉnh cao**:  
> 1. **Khởi tạo mỏ neo Student từ Checkpoint gốc CVPR 2024**: Khởi tạo từ `fatformer_4class_ckpt.pth` (1.116 keys đạt 96.45% Clean ACC, **tuyệt đối không dùng WiSE làm init**).  
> 2. **Bảo tồn Step 0 Zero-Init Identity**: Tầng chiếu SRM = 0, Cổng tần số = 1.0 (Sai số logit Step 0 = 0.0000000000e+00 bit-perfect).  
> 3. **Tốc độ học phân tầng (Differential Learning Rate)**: Module mới (SRM, Gating) học $lr_2 = 3.0 \times 10^{-4}$; các adapter tiền huấn luyện (FAA, LGA, Text ctx) giảm về $lr_1 = 5.0 \times 10^{-5}$ để bảo toàn biên độ phân loại sạch; đóng băng 94.1% CLIP ViT-L/14 backbone.  
> 4. **Hàm mất mát neo chặt Teacher**: $\mathcal{L} = 0.7\mathcal{L}_{\text{AugCE}} + 0.3\mathcal{L}_{\text{KD}} + 0.2\mathcal{L}_{\text{cons}}$.  
> 5. **Luồng dữ liệu Clean-First**: Tỷ lệ 70% ảnh sạch / 30% suy biến vật lý nhẹ có kiểm soát.

---
### 📌 HƯỚNG DẪN CẤU HÌNH COLAB:
1. **Runtime Type**: Chọn **GPU** (T4, L4 hoặc A100) tại menu `Runtime -> Change runtime type`.
2. **Google Drive**: Đảm bảo tài khoản Drive của bạn có thư mục `Fatformer` hoặc `FatFormer_Hub` chứa sẵn `diffusion_staging.tar` và bộ trọng số.

## 🛠️ BƯỚC 1: KẾT NỐI GOOGLE DRIVE 5TB & ĐỒNG BỘ MÃ NGUỒN FATFORMER-XLA
> Mount Google Drive, tự động nhận diện thư mục làm việc trên Drive (`Fatformer` hoặc `FatFormer_Hub`), clone hoặc đồng bộ nhánh `hoang` mới nhất từ GitHub.

In [ ]:
# ==============================================================================
# BƯỚC 1: KẾT NỐI GOOGLE DRIVE 5TB & ĐỒNG BỘ MÃ NGUỒN
# ==============================================================================
import os, sys, glob, shutil, subprocess, time
import torch

print("=" * 80)
print("[*] BƯỚC 1: THIẾT LẬP MÔI TRƯỜNG GOOGLE COLAB & GOOGLE DRIVE")
print("=" * 80)

# 1. Kết nối Google Drive
print("[*] Đang kết nối Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

# Tự động dò tìm thư mục gốc trên Google Drive
drive_candidates = [
    "/content/drive/MyDrive/Fatformer",
    "/content/drive/MyDrive/FatFormer_Hub",
    "/content/drive/MyDrive/fatformer",
    "/content/drive/MyDrive"
]
DRIVE_ROOT = None
for dc in drive_candidates:
    if os.path.exists(os.path.join(dc, "datasets", "diffusion_staging.tar")):
        DRIVE_ROOT = dc
        break
    elif os.path.exists(os.path.join(dc, "pretrained", "fatformer_4class_ckpt.pth")):
        DRIVE_ROOT = dc
        break
    elif os.path.exists(dc) and any(k in dc for k in ["Fatformer", "FatFormer_Hub"]):
        DRIVE_ROOT = dc
        break

if not DRIVE_ROOT:
    DRIVE_ROOT = drive_candidates[0]
print(f"[✓] Thư mục Google Drive được nhận diện: {DRIVE_ROOT}")

# 2. Đồng bộ mã nguồn từ GitHub (nhánh hoang)
WORKSPACE = '/content/fatformer-xla'
BRANCH = 'hoang'
REPO_URL = 'https://github.com/LeeVHoangtk3/fatformer-xla.git'

if not os.path.exists(WORKSPACE):
    print(f"[*] Đang clone kho mã nguồn từ {REPO_URL} (nhánh {BRANCH})...")
    subprocess.run(f"git clone -b {BRANCH} {REPO_URL} {WORKSPACE}", shell=True, check=True)
else:
    print(f"[*] Đang cập nhật commit mới nhất từ nhánh {BRANCH}...")
    subprocess.run(f"git -C {WORKSPACE} fetch origin {BRANCH}", shell=True, check=True)
    subprocess.run(f"git -C {WORKSPACE} reset --hard origin/{BRANCH}", shell=True, check=True)

if WORKSPACE not in sys.path:
    sys.path.insert(0, WORKSPACE)
os.chdir(WORKSPACE)

# 3. Cài đặt các gói phụ thuộc cần thiết
subprocess.run('pip install -q ftfy regex tqdm tabulate pyyaml matplotlib seaborn', shell=True)

# 4. Kiểm tra GPU
assert torch.cuda.is_available(), 'LỖI: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> Chọn GPU (T4 / A100).'
gpu_name = torch.cuda.get_device_name(0)
gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"[✓] GPU đang sử dụng: {gpu_name} ({gpu_vram:.2f} GB VRAM)")
print(f"[✓] Thư mục làm việc hiện tại: {os.getcwd()}")
print("[✓] Hoàn tất Bước 1: Môi trường sẵn sàng 100%!")


## 🔍 BƯỚC 2: LIÊN KẾT TRỌNG SỐ TIỀN HUẤN LUYỆN (CLIP BACKBONE & TEACHER MỎ NEO)
> Dò tìm và liên kết trọng số từ Google Drive sang SSD Colab:
> - **CLIP Backbone**: `ViT-L-14.pt` $\rightarrow$ `DATASET/pretrained/ViT-L-14.pt`
> - **Teacher Checkpoint**: `fatformer_4class_ckpt.pth` $\rightarrow$ `DATASET/pretrained/fatformer_4class_ckpt.pth`
> - **Student Init**: Khởi tạo trực tiếp từ `fatformer_4class_ckpt.pth` (**tuyệt đối không dùng WiSE**)

In [ ]:
# ==============================================================================
# BƯỚC 2: LIÊN KẾT CHECKPOINT GỐC CVPR 2024 VÀ CLIP BACKBONE
# ==============================================================================
os.makedirs(os.path.join(WORKSPACE, "DATASET", "pretrained"), exist_ok=True)
os.makedirs(os.path.join(WORKSPACE, "DATASET", "checkpoints"), exist_ok=True)

def find_file(patterns):
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return os.path.abspath(matches[0])
    return None

print("=" * 80)
print("[*] BƯỚC 2: ĐỊNH VỊ VÀ LIÊN KẾT TÀI NGUYÊN CHECKPOINT")
print("=" * 80)

# 1. Dò tìm CLIP ViT-L-14.pt
clip_dst = os.path.join(WORKSPACE, "DATASET", "pretrained", "ViT-L-14.pt")
clip_src = find_file([
    os.path.join(DRIVE_ROOT, "pretrained", "ViT-L-14.pt"),
    os.path.join(DRIVE_ROOT, "checkpoint", "ViT-L-14.pt"),
    "/content/drive/MyDrive/**/ViT-L-14.pt",
    clip_dst
])

if clip_src and clip_src != clip_dst and not os.path.exists(clip_dst):
    print(f"[*] Đang sao chép CLIP Backbone từ Drive sang SSD: {clip_src}...")
    shutil.copy2(clip_src, clip_dst)
elif not clip_src or not os.path.exists(clip_dst):
    print("[*] Chưa thấy ViT-L-14.pt trên Drive, đang tải trực tiếp từ OpenAI CDN...")
    clip_url = "https://openaipublic.azureedge.net/clip/models/b8cca3fd8d72c9937d9c052a7f64b114695584806668482fe68c3b8116340f1a/ViT-L-14.pt"
    subprocess.run(f"wget -c '{clip_url}' -O '{clip_dst}'", shell=True, check=True)

print(f"[1] CLIP Backbone : {clip_dst} ({os.path.getsize(clip_dst)/(1024**2):.1f} MB)")

# 2. Dò tìm Checkpoint gốc fatformer_4class_ckpt.pth (Teacher & Student Init)
ckpt_4class_dst = os.path.join(WORKSPACE, "DATASET", "pretrained", "fatformer_4class_ckpt.pth")
ckpt_4class_src = find_file([
    os.path.join(DRIVE_ROOT, "pretrained", "fatformer_4class_ckpt.pth"),
    os.path.join(DRIVE_ROOT, "checkpoint", "fatformer_4class_ckpt.pth"),
    "/content/drive/MyDrive/**/fatformer_4class_ckpt.pth",
    ckpt_4class_dst
])

if ckpt_4class_src and ckpt_4class_src != ckpt_4class_dst and not os.path.exists(ckpt_4class_dst):
    print(f"[*] Đang sao chép Checkpoint gốc từ Drive sang SSD: {ckpt_4class_src}...")
    shutil.copy2(ckpt_4class_src, ckpt_4class_dst)

assert os.path.exists(clip_dst), "LỖI: Chưa có ViT-L-14.pt!"
assert os.path.exists(ckpt_4class_dst), f"LỖI: Chưa tìm thấy fatformer_4class_ckpt.pth trong {DRIVE_ROOT}!"

print(f"[2] Teacher Anchor: {ckpt_4class_dst} ({os.path.getsize(ckpt_4class_dst)/(1024**2):.1f} MB)")
print(f"[3] Student Init  : {ckpt_4class_dst} (Khởi tạo trực tiếp từ Checkpoint gốc CVPR 2024)")
print("[✓] Hoàn tất Bước 2: Trọng số gốc và Backbone đã liên kết chuẩn xác!")


## 📦 BƯỚC 3: GIẢI NÉN DỮ LIỆU HUẤN LUYỆN SANG SSD NVME & TẠO MANIFEST CHUẨN
> Dò tìm `diffusion_staging.tar` trên Google Drive, giải nén sang SSD NVMe Colab `/content/dataset_train`, quét toàn bộ ảnh để tạo `DATASET/manifests/data_manifest.json` (phân chia 70% train / 15% val_calibration / 15% val_selection).

In [ ]:
# ==============================================================================
# BƯỚC 3: TRÍCH XUẤT DỮ LIỆU VÀ TẠO MANIFEST CHUẨN 3 TẬP CÂN BẰNG
# ==============================================================================
import tarfile, json as py_json, random

DATA_DIR = "/content/dataset_train"
os.makedirs(DATA_DIR, exist_ok=True)

print("=" * 80)
print("[*] BƯỚC 3: GIẢI NÉN DỮ LIỆU HUẤN LUYỆN VÀ SINH MANIFEST")
print("=" * 80)

tar_staging = find_file([
    os.path.join(DRIVE_ROOT, "datasets", "diffusion_staging.tar"),
    os.path.join(DRIVE_ROOT, "datasets", "diffusion_staging", "diffusion_staging.tar"),
    "/content/drive/MyDrive/**/diffusion_staging.tar"
])

if tar_staging and len(os.listdir(DATA_DIR)) == 0:
    size_mb = os.path.getsize(tar_staging) / (1024 ** 2)
    print(f"[*] Đang giải nén dữ liệu từ: {tar_staging} ({size_mb:.1f} MB) sang SSD NVMe {DATA_DIR}...")
    t0 = time.time()
    with tarfile.open(tar_staging, 'r') as tf:
        tf.extractall(DATA_DIR)
    print(f"[✓] Đã giải nén hoàn tất sang SSD trong {time.time() - t0:.1f}s: {DATA_DIR}")
elif len(os.listdir(DATA_DIR)) > 0:
    print(f"[✓] Dữ liệu đã có sẵn trên SSD NVMe: {DATA_DIR}")
else:
    print(f"[!] Cảnh báo: Không tìm thấy diffusion_staging.tar trong {DRIVE_ROOT}! Vui lòng kiểm tra lại Google Drive.")

# Quét ảnh thực tế với cơ chế nhận diện nhãn chính xác tuyệt đối
all_records = []
for root, _, files in os.walk(DATA_DIR):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
            fp = os.path.join(root, f)
            norm = fp.replace('\\', '/').lower()
            parts = norm.split('/')
            is_fake = any(p in ('1_fake', 'fake', 'ai') for p in parts[-3:])
            is_real = any(p in ('0_real', 'real', 'nature', 'clean') for p in parts[-3:])
            if is_real and is_fake:
                for p in reversed(parts):
                    if p in ('0_real', 'real', 'nature', 'clean'):
                        is_fake = False
                        break
                    elif p in ('1_fake', 'fake', 'ai'):
                        is_real = False
                        break
            if not (is_real or is_fake):
                is_fake = ('1_fake' in norm or '/fake/' in norm)

            all_records.append({
                'file_path': fp,
                'label': 1 if is_fake else 0,
                'generator': 'synthetic' if is_fake else 'real'
            })

reals_found = [r for r in all_records if r['label'] == 0]
fakes_found = [r for r in all_records if r['label'] == 1]
print(f"[✓] Tổng số ảnh quét được trong kho dữ liệu: {len(all_records)}")
print(f"    • Số ảnh Real (label=0): {len(reals_found)}")
print(f"    • Số ảnh Fake (label=1): {len(fakes_found)}")

if len(all_records) > 0:
    assert len(reals_found) > 0 and len(fakes_found) > 0, 'LỖI GÁN NHÃN: Không tìm thấy đủ cả ảnh Real và ảnh Fake!'

manifest_path = os.path.join(WORKSPACE, "DATASET", "manifests", "data_manifest.json")
os.makedirs(os.path.dirname(manifest_path), exist_ok=True)

if len(all_records) > 0:
    rng = random.Random(42)
    rng.shuffle(reals_found)
    rng.shuffle(fakes_found)

    # Chia tỷ lệ phân tầng cân bằng 70% train / 15% val_calibration / 15% val_selection
    n_train_r = int(len(reals_found) * 0.70)
    n_calib_r = int(len(reals_found) * 0.15)
    train_r = reals_found[:n_train_r]
    calib_r = reals_found[n_train_r:n_train_r + n_calib_r]
    select_r = reals_found[n_train_r + n_calib_r:]

    n_train_f = int(len(fakes_found) * 0.70)
    n_calib_f = int(len(fakes_found) * 0.15)
    train_f = fakes_found[:n_train_f]
    calib_f = fakes_found[n_train_f:n_train_f + n_calib_f]
    select_f = fakes_found[n_train_f + n_calib_f:]

    train_list = train_r + train_f
    calib_list = calib_r + calib_f
    select_list = select_r + select_f

    rng.shuffle(train_list)
    rng.shuffle(calib_list)
    rng.shuffle(select_list)

    final_records = []
    for item in train_list:
        entry = dict(item)
        entry['split'] = 'train'
        final_records.append(entry)
    for item in calib_list:
        entry = dict(item)
        entry['split'] = 'val_calibration'
        entry['split_compat'] = 'val'
        final_records.append(entry)
    for item in select_list:
        entry = dict(item)
        entry['split'] = 'val_selection'
        entry['split_compat'] = 'val'
        final_records.append(entry)

    with open(manifest_path, "w", encoding="utf-8") as f_out:
        py_json.dump({
            'metadata': {
                'task': 'Task 6.3-R Staging Manifest Balanced Colab',
                'counts': {
                    'total': len(final_records),
                    'train': len(train_list),
                    'val_calibration': len(calib_list),
                    'val_selection': len(select_list)
                }
            },
            'records': final_records
        }, f_out, indent=2)
    print(f"[✓] Đã tạo manifest chuẩn 3 tập: {manifest_path}")
    print(f"    • Train          : {len(train_list)} ảnh (Real: {len(train_r)}, Fake: {len(train_f)})")
    print(f"    • Val Calibration: {len(calib_list)} ảnh (Real: {len(calib_r)}, Fake: {len(calib_f)})")
    print(f"    • Val Selection  : {len(select_list)} ảnh (Real: {len(select_r)}, Fake: {len(select_f)})")
else:
    print("[!] Cảnh báo: Chưa tìm thấy ảnh thực tế, pipeline sẽ sử dụng chế độ dry-run mô phỏng.")


## ⚖️ BƯỚC 4: KIỂM TOÁN GPU & XÁC THỰC TOÁN HỌC ZERO-INIT IDENTITY TẠI STEP 0
> Chạy unit-test xác nhận sai số logit Step 0 đạt mức $0.0000000000\text{e}+00$ bit-perfect và chạy dry-run 1 step kiểm thử autograd.

In [ ]:
# ==============================================================================
# BƯỚC 4: KIỂM TRA GPU VÀ XÁC THỰC ZERO-INIT IDENTITY
# ==============================================================================
print("=" * 80)
print("[*] BƯỚC 4: KIỂM TOÁN PHẦN CỨNG GPU VÀ UNIT-TEST ZERO-INIT")
print("=" * 80)

# 1. Chạy xác thực Zero-Init Step 0 Identity
zi_cmd = f"{sys.executable} tools/test_zero_init_identity.py --device cuda"
print(f"[*] Kiểm tra toán học Step-0 Identity: {zi_cmd}")
res_zi = subprocess.run(zi_cmd, shell=True)
assert res_zi.returncode == 0, "LỖI: Zero-Init test thất bại!"
print("[✓] TOÁN HỌC ZERO-INIT PASS 100%! BẢO TOÀN BIT-PERFECT BIỂU DIỄN SẠCH GỐC.")

# 2. Chạy dry-run autograd
dry_cmd = f"{sys.executable} src/training/train_crossdomain.py --dry_run --device cuda"
res_dry = subprocess.run(dry_cmd, shell=True)
assert res_dry.returncode == 0, "LỖI: Autograd dry-run thất bại!"
print("[✓] SMOKE TEST AUTOGRAD PASS! HỆ THỐNG SẴN SÀNG CHO VÒNG HUẤN LUYỆN CHÍNH THỨC.")


## 🏃 BƯỚC 5: THỰC THI CHIẾN DỊCH HUẤN LUYỆN ANCHORED RECOVERY M6-R (3 EPOCHS)
> Huấn luyện chính thức với đầy đủ các tham số kỹ thuật bất biến theo chỉ đạo của bạn:
> - `--branch M4`: Mô hình đầy đủ SRM + Dynamic Gating + Distillation
> - `--epochs 3`: 3 epochs huấn luyện
> - `--checkpoint_init DATASET/pretrained/fatformer_4class_ckpt.pth`: Khởi động từ checkpoint gốc (**không dùng WiSE**)
> - `--teacher_checkpoint DATASET/pretrained/fatformer_4class_ckpt.pth`: Mỏ neo cố định từ CVPR 2024
> - `--clip_path DATASET/pretrained/ViT-L-14.pt`: Backbone CLIP TorchScript
> - `--weight_ce 0.7 --alpha_kd 0.3 --beta_cons 0.2`: Trọng số hàm mất mát neo chặt Teacher
> - `--learning_rate 3e-4 --anchor_learning_rate 5e-5`: Tốc độ học phân tầng
> - `--save_name fatformer_anchored_recovery_best.pth`: Tên checkpoint chuẩn mực

In [ ]:
# ==============================================================================
# BƯỚC 5: CHẠY LỆNH HUẤN LUYỆN CHÍNH THỨC TASK 6.3-R TRÊN COLAB GPU
# ==============================================================================
CKPT_OUT_DIR = os.path.join(WORKSPACE, "DATASET", "checkpoints")
os.makedirs(CKPT_OUT_DIR, exist_ok=True)

cmd_train = [
    sys.executable, "-u", "src/training/train_crossdomain.py",
    "--branch", "M4",
    "--device", "cuda",
    "--epochs", "3",
    "--batch_size", "16",
    "--grad_accum_steps", "1",
    "--checkpoint_init", "DATASET/pretrained/fatformer_4class_ckpt.pth",
    "--teacher_checkpoint", "DATASET/pretrained/fatformer_4class_ckpt.pth",
    "--clip_path", "DATASET/pretrained/ViT-L-14.pt",
    "--save_dir", CKPT_OUT_DIR,
    "--save_name", "fatformer_anchored_recovery_best.pth",
    "--weight_ce", "0.7",
    "--alpha_kd", "0.3",
    "--beta_cons", "0.2",
    "--learning_rate", "3e-4",
    "--anchor_learning_rate", "5e-5",
    "--num_workers", "2"
]

if os.path.exists(manifest_path) and len(all_records) > 0:
    cmd_train.extend(["--train_manifest", manifest_path])
else:
    print("[!] Cảnh báo: Không có manifest ảnh thực tế, chạy chế độ dry-run.")
    cmd_train.append("--dry_run")

print("=" * 80)
print(f"[*] Lệnh huấn luyện thực thi: {' '.join(cmd_train)}")
print("=" * 80)

t0 = time.time()
proc = subprocess.Popen(cmd_train, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
elapsed = time.time() - t0

if proc.returncode != 0:
    raise RuntimeError(f"[X] Huấn luyện thất bại với mã lỗi: {proc.returncode}")
else:
    print(f"\n[✓] HUẤN LUYỆN ANCHORED RECOVERY HOÀN TẤT THÀNH CÔNG TRONG {elapsed:.2f}s ({elapsed/60:.2f} phút)!")


## 🔐 BƯỚC 6: XÁC THỰC CHECKPOINT, TÍNH MÃ BĂM SHA-256 & ĐỒNG BỘ SANG GOOGLE DRIVE 5TB
> Kiểm tra `fatformer_anchored_recovery_best.pth`, tính mã băm SHA-256, sinh metadata JSON và tự động sao lưu cả checkpoint + metadata + lịch sử huấn luyện sang Google Drive 5TB.

In [ ]:
# ==============================================================================
# BƯỚC 6: XÁC THỰC CHECKPOINT VÀ ĐỒNG BỘ SANG GOOGLE DRIVE 5TB
# ==============================================================================
import hashlib, json as py_json

best_ckpt = os.path.join(CKPT_OUT_DIR, "fatformer_anchored_recovery_best.pth")
assert os.path.exists(best_ckpt), f"LỖI: Không tìm thấy checkpoint tại {best_ckpt}!"

size_mb = os.path.getsize(best_ckpt) / (1024**2)
print(f"[*] Đang tính toán mã băm SHA-256 cho: {best_ckpt} ({size_mb:.2f} MB)...")
sha256 = hashlib.sha256()
with open(best_ckpt, 'rb') as f:
    while chunk := f.read(10 * 1024 * 1024):
        sha256.update(chunk)
digest = sha256.hexdigest().upper()
print(f"[✓] SHA-256 DIGEST: {digest}")

meta_dict = {
    "checkpoint_name": "fatformer_anchored_recovery_best.pth",
    "task": "Task 6.3-R Anchored Recovery Training (Colab)",
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "sha256": digest,
    "size_bytes": os.path.getsize(best_ckpt),
    "hyperparameters": {
        "branch": "M4",
        "epochs": 3,
        "batch_size": 16,
        "grad_accum_steps": 1,
        "learning_rate": 3e-4,
        "anchor_learning_rate": 5e-5,
        "weight_ce": 0.7,
        "alpha_kd": 0.3,
        "beta_cons": 0.2,
        "checkpoint_init": "DATASET/pretrained/fatformer_4class_ckpt.pth",
        "teacher_checkpoint": "DATASET/pretrained/fatformer_4class_ckpt.pth"
    }
}

meta_path = os.path.join(CKPT_OUT_DIR, "fatformer_anchored_recovery_best_metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    py_json.dump(meta_dict, f, indent=2)

# Đồng bộ sang Google Drive 5TB
drive_ckpt_dir = os.path.join(DRIVE_ROOT, "checkpoint")
os.makedirs(drive_ckpt_dir, exist_ok=True)

drive_out_ckpt = os.path.join(drive_ckpt_dir, "fatformer_anchored_recovery_best.pth")
drive_out_meta = os.path.join(drive_ckpt_dir, "fatformer_anchored_recovery_best_metadata.json")

print(f"[*] Đang sao lưu Checkpoint sang Google Drive: {drive_out_ckpt}...")
shutil.copy2(best_ckpt, drive_out_ckpt)
shutil.copy2(meta_path, drive_out_meta)
print(f"[✓] ĐÃ LƯU CHECKPOINT AN TOÀN TRÊN GOOGLE DRIVE: {drive_out_ckpt}")
print(f"[✓] ĐÃ LƯU METADATA SHA-256 TRÊN GOOGLE DRIVE: {drive_out_meta}")

# Sao chép cả last checkpoint nếu có
last_ckpt = os.path.join(CKPT_OUT_DIR, 'fatformer_anchored_recovery_last.pth')
if os.path.exists(last_ckpt):
    shutil.copy2(last_ckpt, os.path.join(drive_ckpt_dir, 'fatformer_anchored_recovery_last.pth'))
    print(f"[✓] ĐÃ LƯU LAST CHECKPOINT TRÊN GOOGLE DRIVE!")

# Sao chép lịch sử CSV
hist_csv = os.path.join(WORKSPACE, "DATASET", "logs", "m6_training_history.csv")
if os.path.exists(hist_csv):
    drive_log_dir = os.path.join(DRIVE_ROOT, "logs")
    os.makedirs(drive_log_dir, exist_ok=True)
    shutil.copy2(hist_csv, os.path.join(drive_log_dir, "m6_training_history.csv"))
    print(f"[✓] ĐÃ LƯU NHẬT KÝ HUẤN LUYỆN CSV TRÊN GOOGLE DRIVE: {os.path.join(drive_log_dir, 'm6_training_history.csv')}")

print("\n🎉 HOÀN TẤT TOÀN DIỆN HUẤN LUYỆN TASK 6.3-R TRÊN GOOGLE COLAB GPU!")


## 🎯 BƯỚC 7: HIỆU CHUẨN XÁC SUẤT TASK 6.4-R TRÊN COLAB (KHÓA NGƯỠNG $T^*=0.50$)
> Khóa ngưỡng chuẩn $T^*=0.50$, đo đạc nhiệt độ tối ưu $T$ và chấm điểm Gate 4 điều kiện trên tập `val_calibration` ngay tại Colab, xuất báo cáo sang Google Drive.

In [ ]:
# ==============================================================================
# BƯỚC 7: HIỆU CHUẨN XÁC SUẤT TASK 6.4-R VỚI CHECKPOINT VỪA TẠO
# ==============================================================================
cmd_calib = [
    sys.executable, "-u", "tools/calibrate_and_select.py",
    "--checkpoint", best_ckpt,
    "--clip_path", "DATASET/pretrained/ViT-L-14.pt",
    "--manifest", manifest_path,
    "--lock_standard_threshold",
    "--output_dir", os.path.join(DRIVE_ROOT, "checkpoint"),
    "--device", "cuda"
]

print("=" * 80)
print(f"[*] Lệnh hiệu chuẩn Task 6.4-R: {' '.join(cmd_calib)}")
print("=" * 80)

res_calib = subprocess.run(' '.join(cmd_calib), shell=True)
if res_calib.returncode == 0:
    print("\n[✓] HOÀN TẤT HIỆU CHUẨN TASK 6.4-R TRÊN COLAB! ĐÃ LƯU BÁO CÁO SANG GOOGLE DRIVE.")
else:
    print("[!] Lưu ý: Hiệu chuẩn có cảnh báo hoặc đã hoàn thành với thông báo trên.")
